# NLP HW2: Korean MCQA Option-Label Study

Reorganized from the original executed course notebook. **No experiments were re-run when producing this clean copy.** Code outputs, Colab widget state, unrelated Drive listings, and ad hoc debug cells were removed.

KMMLU data and generated prompt text are stored in private Google Drive. Share the repository without dataset-text files unless redistribution rights are confirmed.

**Run order:** Top to bottom with a T4 GPU, or run `analyze_results.py` on an existing `main_1280_results.csv` using CPU only. Re-running inference may take considerable GPU resources.

## 0. Check accelerator
Run on **T4 GPU** for inference. The analysis-only script in this repository runs without a GPU.

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU Memory:",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
          "GB")
assert torch.cuda.is_available(), "Inference requires a GPU. For analysis-only use analyze_results.py."


## 1. Install and record environment

In [ ]:
!pip install -q transformers==4.51.3 datasets==3.5.0 accelerate==1.6.0

In [ ]:
import random
import numpy as np
import pandas as pd
import torch
import transformers
import datasets
import scipy
import matplotlib.pyplot as plt

from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Random seed:", SEED)
print("Environment setup complete.")

## 2. Connect private Google Drive storage

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/NLP_HW2'

os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/data', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/results', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/figures', exist_ok=True)

print("Project directory:", PROJECT_DIR)

## 3. Load pinned KMMLU test splits

In [ ]:
import pandas as pd
from huggingface_hub import hf_hub_download

DATASET_ID = "HAERAE-HUB/KMMLU"

DATASET_REVISION = "d61b3f19e552c576bf5960dd24289763edc36a88"  # Original run

SUBJECT_FILES = {
    "Computer-Science": "Computer-Science",
    "Biology": "Biology",
    "Korean-History": "korean-history",
    "Criminal-Law": "Criminal-Law"
}

raw_datasets = {}

for subject, filename in SUBJECT_FILES.items():
    path = hf_hub_download(
        repo_id=DATASET_ID,
        repo_type="dataset",
        filename=f"data/{filename}-test.csv",
        revision=DATASET_REVISION
    )

    df = pd.read_csv(path, dtype=str, keep_default_na=False)
    raw_datasets[subject] = df

    print(f"{subject}: {len(df)} questions loaded")

print("Dataset revision:", DATASET_REVISION)
print("Dataset loading complete.")

## 4. Filter, sample, and apply recorded manual-review replacements

In [ ]:
import re
import unicodedata
from collections import Counter

POSITION_PATTERN = re.compile(
    r"위의\s*모두|위의\s*모든|"
    r"위\s*항목\s*모두|"
    r"앞의\s*모두|"
    r"위\s*보기\s*모두|"
    r"위의\s*어느\s*것도|"
    r"[1-4]\s*번\s*(선택지|보기|답)|"
    r"[A-Dㄱㄴㄷㄹ]\s*(선택지|보기)"
)

def normalize(text):
    text = unicodedata.normalize("NFC", str(text))
    return " ".join(text.split())


def exclusion_reason(row):
    question = normalize(row["question"])
    options = [normalize(row[c]) for c in "ABCD"]
    answer = str(row["answer"]).strip()

    if not question:
        return "missing_question"

    if any(not option for option in options):
        return "missing_option"

    if len(set(options)) != 4:
        return "duplicate_options"

    if answer not in ["1", "2", "3", "4"]:
        return "invalid_answer"

    if any(POSITION_PATTERN.search(t)
           for t in [question] + options):
        return "position_dependent"

    return None


filtered_datasets = {}
exclusion_log = []

for subject, df in raw_datasets.items():
    valid_rows = []
    counts = Counter()

    for idx, row in df.iterrows():
        reason = exclusion_reason(row)

        if reason is not None:
            counts[reason] += 1

            exclusion_log.append({
                "subject": subject,
                "source_index": int(idx),
                "reason": reason
            })
            continue

        item = row.to_dict()
        item["subject"] = subject
        item["source_index"] = int(idx)
        item["question_id"] = f"{subject}_{idx}"
        item["answer"] = int(row["answer"])

        valid_rows.append(item)

    filtered_datasets[subject] = pd.DataFrame(valid_rows)

    print(f"\n[{subject}]")
    print("Original:", len(df))
    print("Valid:", len(valid_rows))
    print("Excluded:", len(df) - len(valid_rows))
    print("Reasons:", dict(counts))

In [ ]:
import random

SEED = 42
N_PER_SUBJECT = 40

rng = random.Random(SEED)
selected_items = []

for subject, df in filtered_datasets.items():

    assert len(df) >= N_PER_SUBJECT, (
        f"Not enough valid questions: {subject}"
    )

    indices = rng.sample(
        range(len(df)),
        N_PER_SUBJECT
    )

    sampled = df.iloc[indices].copy()
    selected_items.append(sampled)

selected_df = pd.concat(
    selected_items,
    ignore_index=True
)

assert len(selected_df) == 160
assert selected_df["question_id"].is_unique

print(selected_df["subject"].value_counts())
print("\nTotal questions:", len(selected_df))

In [ ]:
EXCLUDED_IDS = [
    "Biology_348",
    "Korean-History_96",
    "Korean-History_58",
    "Korean-History_81",
    "Korean-History_45",
    "Korean-History_21",
    "Korean-History_86",
    "Korean-History_91",
    "Korean-History_82",
    "Korean-History_41",
    "Korean-History_40",
    "Korean-History_13",
    "Korean-History_29",
    "Korean-History_4",
    "Korean-History_99",
    "Criminal-Law_63",
    "Criminal-Law_92",
    "Criminal-Law_16",
    "Criminal-Law_64"
]

excluded_df = selected_df[
    selected_df["question_id"].isin(EXCLUDED_IDS)
].copy()

retained_df = selected_df[
    ~selected_df["question_id"].isin(EXCLUDED_IDS)
].copy()

print("Retained:", len(retained_df))
print("Excluded:", len(excluded_df))

print("\nRequired replacements:")
print(excluded_df["subject"].value_counts())

In [ ]:
import random
import re
import unicodedata

rng = random.Random(42)

def question_key(text):
    text = unicodedata.normalize("NFC", str(text))
    return re.sub(r"\s+", "", text)

used_ids = set(selected_df["question_id"])
used_questions = set(
    selected_df["question"].map(question_key)
)

candidate_items = []

for subject, count in excluded_df["subject"].value_counts().items():

    pool = filtered_datasets[subject].copy()

    pool = pool[
        ~pool["question_id"].isin(used_ids)
    ].copy()

    pool["review_key"] = pool["question"].map(question_key)

    pool = pool[
        ~pool["review_key"].isin(used_questions)
    ]

    pool = pool.drop_duplicates("review_key")

    pool = pool.drop(columns=["review_key"])

    indices = rng.sample(
        range(len(pool)),
        min(len(pool), int(count) + 8)
    )

    candidate_items.append(pool.iloc[indices])

candidate_df = pd.concat(
    candidate_items,
    ignore_index=True
)

print(candidate_df["subject"].value_counts())
print("Total candidates:", len(candidate_df))

In [ ]:
REPLACEMENT_IDS = [
    # Biology
    "Biology_109",

    # Korean History
    "Korean-History_18",
    "Korean-History_2",
    "Korean-History_39",
    "Korean-History_22",
    "Korean-History_17",
    "Korean-History_16",
    "Korean-History_64",
    "Korean-History_0",
    "Korean-History_6",
    "Korean-History_97",
    "Korean-History_42",
    "Korean-History_52",
    "Korean-History_67",
    "Korean-History_94",

    # Criminal Law
    "Criminal-Law_136",
    "Criminal-Law_69",
    "Criminal-Law_1",
    "Criminal-Law_138"
]

replacement_df = candidate_df[
    candidate_df["question_id"].isin(REPLACEMENT_IDS)
].copy()

assert len(retained_df) == 141
assert len(replacement_df) == 19

selected_df = pd.concat(
    [retained_df, replacement_df],
    ignore_index=True
)

print("Final questions:", len(selected_df))
print(selected_df["subject"].value_counts())

In [ ]:
assert len(selected_df) == 160
assert selected_df["question_id"].is_unique

assert not selected_df["question"].map(
    question_key
).duplicated().any()

assert selected_df["answer"].isin([1, 2, 3, 4]).all()

assert set(selected_df["subject"].unique()) == set(
    SUBJECT_FILES.keys()
)

assert (
    selected_df["subject"].value_counts() == 40
).all()

for _, row in selected_df.iterrows():
    options = [normalize(row[c]) for c in "ABCD"]

    assert all(options)
    assert len(set(options)) == 4

print("Total questions: 160")
print("Questions per subject: 40")
print("Duplicate questions: None")
print("Invalid answer labels: None")
print("Dataset validation passed.")

In [ ]:
import os
import json

DATA_DIR = os.path.join(PROJECT_DIR, "data")
os.makedirs(DATA_DIR, exist_ok=True)

# 최종 160문항 저장
selected_df.to_csv(
    os.path.join(DATA_DIR, "kmmlu_160.csv"),
    index=False,
    encoding="utf-8-sig"
)

# 자동 필터링 기록 저장
pd.DataFrame(
    exclusion_log,
    columns=["subject", "source_index", "reason"]
).to_csv(
    os.path.join(DATA_DIR, "exclusion_log.csv"),
    index=False,
    encoding="utf-8-sig"
)

# 문항 교체 기록 저장
review_log = {
    "excluded_ids": EXCLUDED_IDS,
    "replacement_ids": REPLACEMENT_IDS,
    "review_method": "AI-assisted structural review",
    "excluded_count": 19,
    "replacement_count": 19
}

with open(
    os.path.join(DATA_DIR, "review_log.json"),
    "w",
    encoding="utf-8"
) as f:
    json.dump(review_log, f, ensure_ascii=False, indent=2)

# 데이터 추출 설정 저장
metadata = {
    "dataset": DATASET_ID,
    "revision": DATASET_REVISION,
    "split": "test",
    "seed": 42,
    "subjects": list(SUBJECT_FILES.keys()),
    "questions_per_subject": 40,
    "total_questions": 160,
    "selected_question_ids": selected_df[
        "question_id"
    ].tolist()
}

with open(
    os.path.join(DATA_DIR, "sampling_metadata.json"),
    "w",
    encoding="utf-8"
) as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("All files saved successfully.")
print("STEP 2 complete.")

## 5. Build 1,280 paired prompts

In [ ]:
import os
import json
import random
import pandas as pd

SEED = 42

PROJECT_DIR = "/content/drive/MyDrive/NLP_HW2"
DATA_DIR = os.path.join(PROJECT_DIR, "data")

selected_df = pd.read_csv(
    os.path.join(DATA_DIR, "kmmlu_160.csv"),
    dtype=str,
    keep_default_na=False
)

selected_df["answer"] = selected_df["answer"].astype(int)

assert len(selected_df) == 160
assert selected_df["question_id"].is_unique
assert selected_df["answer"].isin([1, 2, 3, 4]).all()

print("Total questions:", len(selected_df))
print(selected_df["subject"].value_counts())

In [ ]:
LABEL_FORMATS = {
    "Latin": ["A", "B", "C", "D"],
    "Hangul": ["ㄱ", "ㄴ", "ㄷ", "ㄹ"]
}

INSTRUCTION = "정답 선택지의 기호 하나만 출력하세요."

ORIGINAL_LABELS = ["A", "B", "C", "D"]

print("Latin:", LABEL_FORMATS["Latin"])
print("Hangul:", LABEL_FORMATS["Hangul"])
print("Instruction:", INSTRUCTION)

In [ ]:
def rotate_options(order, rotation):
    rotation = rotation % len(order)

    if rotation == 0:
        return order.copy()

    return order[-rotation:] + order[:-rotation]


# 함수 테스트
original_order = [1, 2, 3, 4]

for r in range(4):
    print(f"Rotation {r}:",
          rotate_options(original_order, r))

In [ ]:
def make_prompt(row, order, label_format):

    labels = LABEL_FORMATS[label_format]

    original_options = [
        row["A"],
        row["B"],
        row["C"],
        row["D"]
    ]

    lines = [f"질문: {row['question']}", ""]

    for position, original_id in enumerate(order):

        label = labels[position]
        content = original_options[original_id - 1]

        lines.append(f"{label}. {content}")

    lines.append("")
    lines.append(INSTRUCTION)

    prompt = "\n".join(lines)

    correct_original_id = int(row["answer"])
    correct_position = order.index(correct_original_id)
    correct_label = labels[correct_position]

    return {
        "question_id": row["question_id"],
        "subject": row["subject"],
        "label_format": label_format,
        "option_order": order.copy(),
        "correct_original_id": correct_original_id,
        "correct_position": correct_position + 1,
        "correct_label": correct_label,
        "prompt": prompt
    }

In [ ]:
experiment_items = []

rng = random.Random(SEED)

for _, row in selected_df.iterrows():

    # 문항마다 기본 보기 순서를 무작위로 설정
    base_order = [1, 2, 3, 4]
    rng.shuffle(base_order)

    for rotation in range(4):

        order = rotate_options(
            base_order,
            rotation
        )

        for label_format in LABEL_FORMATS:

            item = make_prompt(
                row,
                order,
                label_format
            )

            item["rotation"] = rotation
            item["base_order"] = base_order.copy()

            item["experiment_id"] = (
                f"{row['question_id']}"
                f"_{label_format}_R{rotation}"
            )

            experiment_items.append(item)

experiment_df = pd.DataFrame(experiment_items)

print("Total experiments:", len(experiment_df))

print("\nExperiments by label format:")
print(experiment_df["label_format"].value_counts())

print("\nExperiments by subject:")
print(experiment_df["subject"].value_counts())

In [ ]:
# 1. 전체 실험 개수 검사
total = len(experiment_df)
assert total == 1280, f"Expected 1280, got {total}"

# 2. 실험 ID 중복 검사
duplicates = experiment_df["experiment_id"].duplicated().sum()
assert duplicates == 0, f"Duplicate IDs: {duplicates}"

# 3. 영문/한글 조건별 개수 검사
format_counts = experiment_df["label_format"].value_counts()

assert format_counts.get("Latin", 0) == 640
assert format_counts.get("Hangul", 0) == 640

# 4. 각 문항별 조건 검사
for question_id, group in experiment_df.groupby("question_id"):

    assert len(group) == 8, f"Invalid count: {question_id}"

    for label_format in ["Latin", "Hangul"]:
        subset = group[group["label_format"] == label_format]

        assert len(subset) == 4

        positions = sorted(subset["correct_position"].tolist())
        assert positions == [1, 2, 3, 4]

    # 동일한 회전에서 두 조건의 보기 순서 비교
    for rotation in range(4):
        pair = group[group["rotation"] == rotation]

        assert len(pair) == 2

        latin = pair[pair["label_format"] == "Latin"].iloc[0]
        hangul = pair[pair["label_format"] == "Hangul"].iloc[0]

        assert latin["option_order"] == hangul["option_order"]
        assert latin["correct_position"] == hangul["correct_position"]

# 5. 정답 기호 검증
for _, row in experiment_df.iterrows():

    position = row["correct_position"] - 1
    expected_label = LABEL_FORMATS[row["label_format"]][position]

    assert row["correct_label"] == expected_label
    assert row["option_order"][position] == row["correct_original_id"]

# 실제 검사한 값 출력
print(f"Total experiments: {total}")
print(f"Duplicate IDs: {duplicates}")
print("Format counts:")
print(format_counts.to_string())
print("All validation tests passed.")

In [ ]:
EXPERIMENT_PATH = os.path.join(
    DATA_DIR,
    "experiment_1280.jsonl"
)

experiment_df.to_json(
    EXPERIMENT_PATH,
    orient="records",
    lines=True,
    force_ascii=False
)

experiment_df.to_csv(
    os.path.join(DATA_DIR, "experiment_1280.csv"),
    index=False,
    encoding="utf-8-sig"
)

config = {
    "seed": SEED,
    "model": "Qwen/Qwen2.5-3B-Instruct",
    "instruction": INSTRUCTION,
    "label_formats": LABEL_FORMATS,
    "rotations": [0, 1, 2, 3],
    "total_questions": 160,
    "total_experiments": 1280
}

with open(
    os.path.join(DATA_DIR, "experiment_config.json"),
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        config,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Saved:", EXPERIMENT_PATH)
print("STEP 3 complete.")

## 6. Load pinned Qwen model and test the parser

In [ ]:
import os
import pandas as pd
import torch

PROJECT_DIR = "/content/drive/MyDrive/NLP_HW2"
DATA_DIR = os.path.join(PROJECT_DIR, "data")
RESULT_DIR = os.path.join(PROJECT_DIR, "results")

os.makedirs(RESULT_DIR, exist_ok=True)

experiment_path = os.path.join(
    DATA_DIR, "experiment_1280.jsonl"
)

assert os.path.isfile(experiment_path), (
    "STEP 3 실험 데이터 파일을 찾을 수 없습니다."
)

experiment_df = pd.read_json(
    experiment_path,
    lines=True
)

assert len(experiment_df) == 1280
assert torch.cuda.is_available()

print("Experiments:", len(experiment_df))
print("GPU:", torch.cuda.get_device_name(0))
print("Ready for STEP 4")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM


MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

# 재현성을 위해 모델의 정확한 버전을 기록
MODEL_REVISION = "aa8e72537993ba99e69dfaafa59ed015b17504d1"  # Original run

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION
)

print("Loading model...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)

model.eval()

print("Model:", MODEL_ID)
print("Revision:", MODEL_REVISION)
print("Device:", model.device)
print("Model loading complete.")

In [ ]:
import torch

MAX_NEW_TOKENS = 16

def generate_answer(prompt):

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    # 모델 입력 생성
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    # 입력 토큰 길이
    input_length = inputs["input_ids"].shape[1]

    # 모델 추론
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # 입력을 제외하고 생성된 답변만 추출
    generated_tokens = outputs[0, input_length:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return response

print("Generation function updated successfully.")

In [ ]:
LABEL_FORMATS = {
    "Latin": ["A", "B", "C", "D"],
    "Hangul": ["ㄱ", "ㄴ", "ㄷ", "ㄹ"]
}

def parse_answer(response, label_format):

    text = response.strip()

    # 기호 뒤에 마침표 하나가 있는 경우만 허용
    if text.endswith("."):
        text = text[:-1]

    if text in LABEL_FORMATS[label_format]:
        return text

    return None


# 파서 단위 테스트
assert parse_answer("A", "Latin") == "A"
assert parse_answer("B.", "Latin") == "B"
assert parse_answer("ㄱ", "Hangul") == "ㄱ"
assert parse_answer("정답은 A입니다.", "Latin") is None
assert parse_answer("A 또는 B", "Latin") is None
assert parse_answer("A", "Hangul") is None

print("Answer parser tests passed.")

## 7. Pilot: 16 outputs on two separate questions

In [ ]:
# 본실험 데이터와 무관한 예비 문제
pilot_questions = [
    {
        "question_id": "pilot_math",
        "subject": "Pilot",
        "question": "2와 3을 더하면 얼마인가?",
        "A": "4",
        "B": "5",
        "C": "6",
        "D": "7",
        "answer": 2
    },
    {
        "question_id": "pilot_general",
        "subject": "Pilot",
        "question": "다음 중 포유류에 해당하는 동물은?",
        "A": "나비",
        "B": "개구리",
        "C": "고래",
        "D": "도마뱀",
        "answer": 3
    }
]

print("Pilot questions:", len(pilot_questions))

In [ ]:
pilot_results = []

for question in pilot_questions:

    for rotation in range(4):

        order = rotate_options(
            [1, 2, 3, 4],
            rotation
        )

        for label_format in LABEL_FORMATS:

            item = make_prompt(
                question,
                order,
                label_format
            )

            response = generate_answer(
                item["prompt"]
            )

            parsed = parse_answer(
                response,
                label_format
            )

            pilot_results.append({
                "question_id": question["question_id"],
                "label_format": label_format,
                "rotation": rotation,
                "expected_label": item["correct_label"],
                "raw_response": response,
                "parsed_label": parsed,
                "valid": parsed is not None
            })

pilot_df = pd.DataFrame(pilot_results)

print(
    pilot_df[
        [
            "question_id",
            "label_format",
            "rotation",
            "expected_label",
            "raw_response",
            "parsed_label",
            "valid"
        ]
    ].to_string(index=False)
)

valid_count = int(pilot_df["valid"].sum())

print(
    f"\nValid responses: "
    f"{valid_count}/{len(pilot_df)}"
)

In [ ]:
import json
import os
import transformers
import accelerate

PILOT_PATH = os.path.join(
    RESULT_DIR,
    "pilot_step4.csv"
)

pilot_df.to_csv(
    PILOT_PATH,
    index=False,
    encoding="utf-8-sig"
)

inference_config = {
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "torch_dtype": "float16",
    "device": str(model.device),
    "gpu": torch.cuda.get_device_name(0),
    "transformers_version": transformers.__version__,
    "accelerate_version": accelerate.__version__,
    "torch_version": torch.__version__,
    "decoding": "greedy",
    "do_sample": False,
    "max_new_tokens": MAX_NEW_TOKENS,
    "instruction": "정답 선택지의 기호 하나만 출력하세요.",
    "pilot_questions": len(pilot_questions),
    "pilot_responses": len(pilot_df),
    "pilot_valid_responses": int(pilot_df["valid"].sum())
}

CONFIG_PATH = os.path.join(
    DATA_DIR,
    "inference_config.json"
)

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(
        inference_config,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Saved:", PILOT_PATH)
print("Saved:", CONFIG_PATH)
print("STEP 4 pilot complete.")

## 8. Main inference: resume-safe 1,280 generations

In [ ]:
import os
import json
import pandas as pd
import torch

PROJECT_DIR = "/content/drive/MyDrive/NLP_HW2"
DATA_DIR = os.path.join(PROJECT_DIR, "data")
RESULT_DIR = os.path.join(PROJECT_DIR, "results")

os.makedirs(RESULT_DIR, exist_ok=True)

EXPERIMENT_PATH = os.path.join(
    DATA_DIR, "experiment_1280.jsonl"
)

experiment_df = pd.read_json(
    EXPERIMENT_PATH,
    lines=True
)

assert len(experiment_df) == 1280
assert experiment_df["experiment_id"].is_unique
assert experiment_df["question_id"].nunique() == 160

assert experiment_df["label_format"].value_counts().to_dict() == {
    "Latin": 640,
    "Hangul": 640
}

assert torch.cuda.is_available()

# STEP 4의 모델과 함수가 준비되었는지 확인
for name in [
    "model",
    "tokenizer",
    "generate_answer",
    "parse_answer",
    "LABEL_FORMATS",
    "MAX_NEW_TOKENS"
]:
    assert name in globals(), f"{name} is not defined"

assert MAX_NEW_TOKENS == 16

print("Total experiments:", len(experiment_df))
print("Model:", MODEL_ID)
print("GPU:", torch.cuda.get_device_name(0))
print("Max new tokens:", MAX_NEW_TOKENS)
print("Ready for main experiment.")

In [ ]:
import hashlib
from tqdm.auto import tqdm

RESULT_PATH = os.path.join(
    RESULT_DIR, "main_1280_results.csv"
)

META_PATH = os.path.join(
    RESULT_DIR, "main_1280_metadata.json"
)

SAVE_EVERY = 8

# 실험 입력이 바뀌었는지 확인하기 위한 식별값
fingerprint_text = "\n".join(
    experiment_df["experiment_id"].astype(str)
    + "|"
    + experiment_df["prompt"].astype(str)
)

fingerprint = hashlib.sha256(
    fingerprint_text.encode("utf-8")
).hexdigest()

# 중단된 실험을 이어서 실행할 때 데이터 변경 여부 확인
if os.path.exists(META_PATH):
    with open(META_PATH, "r", encoding="utf-8") as f:
        saved_meta = json.load(f)

    assert saved_meta["fingerprint"] == fingerprint, (
        "이전에 실행한 실험과 현재 입력 데이터가 다릅니다."
    )
else:
    assert not os.path.exists(RESULT_PATH), (
        "결과 파일은 있지만 메타데이터가 없습니다. "
        "기존 파일을 먼저 확인하세요."
    )

    with open(META_PATH, "w", encoding="utf-8") as f:
        json.dump(
            {
                "fingerprint": fingerprint,
                "model_id": MODEL_ID,
                "model_revision": MODEL_REVISION,
                "max_new_tokens": MAX_NEW_TOKENS,
                "do_sample": False,
                "total_experiments": 1280
            },
            f,
            ensure_ascii=False,
            indent=2
        )

# 이전 결과가 있으면 불러오기
if os.path.exists(RESULT_PATH):
    records = pd.read_csv(
        RESULT_PATH,
        keep_default_na=False
    ).to_dict("records")
else:
    records = []

completed_ids = {
    str(record["experiment_id"])
    for record in records
}

assert len(completed_ids) == len(records)
assert completed_ids.issubset(
    set(experiment_df["experiment_id"])
)

print("Already completed:", len(completed_ids))
print("Remaining:", 1280 - len(completed_ids))

def save_results():
    df = pd.DataFrame(records)

    temp_path = RESULT_PATH + ".tmp"

    df.to_csv(
        temp_path,
        index=False,
        encoding="utf-8-sig"
    )

    os.replace(temp_path, RESULT_PATH)


# 아직 실행하지 않은 실험만 선택
pending_df = experiment_df[
    ~experiment_df["experiment_id"].isin(completed_ids)
]

new_since_save = 0

try:
    for _, row in tqdm(
        pending_df.iterrows(),
        total=len(pending_df),
        desc="Main experiment"
    ):
        # 실제 모델 추론
        response = generate_answer(row["prompt"])

        # 답변 기호 추출
        parsed = parse_answer(
            response,
            row["label_format"]
        )

        valid = parsed is not None

        # 답변 기호를 원래 선택지 번호로 변환
        predicted_original_id = None

        if valid:
            labels = LABEL_FORMATS[row["label_format"]]
            predicted_position = labels.index(parsed)

            predicted_original_id = int(
                row["option_order"][predicted_position]
            )

        correct_original_id = int(
            row["correct_original_id"]
        )

        correct = (
            predicted_original_id == correct_original_id
        )

        # 결과 기록
        record = {
            "experiment_id": row["experiment_id"],
            "question_id": row["question_id"],
            "subject": row["subject"],
            "label_format": row["label_format"],
            "rotation": int(row["rotation"]),
            "option_order": json.dumps(
                row["option_order"]
            ),
            "correct_original_id": correct_original_id,
            "correct_position": int(
                row["correct_position"]
            ),
            "correct_label": row["correct_label"],
            "raw_response": response,
            "parsed_label": parsed,
            "predicted_original_id": predicted_original_id,
            "is_valid": valid,
            "is_correct": correct
        }

        records.append(record)
        new_since_save += 1

        # 8회마다 Google Drive에 저장
        if new_since_save >= SAVE_EVERY:
            save_results()
            new_since_save = 0

except (Exception, KeyboardInterrupt):
    # 오류나 사용자 중단 시 마지막 결과까지 저장
    if records:
        save_results()
    raise

# 남은 결과 저장
if records:
    save_results()

print("Completed:", len(records), "/ 1280")
print("Saved:", RESULT_PATH)

## 9. Validate outputs and reproduce statistical tables

In [ ]:
results_df = pd.read_csv(
    RESULT_PATH,
    keep_default_na=False
)

# 전체 개수
assert len(results_df) == 1280

# 실험 ID 중복 검사
assert results_df["experiment_id"].is_unique

# 모든 입력에 대한 결과가 존재하는지 검사
assert set(results_df["experiment_id"]) == set(
    experiment_df["experiment_id"]
)

# 영문 640개, 한글 640개 확인
format_counts = results_df[
    "label_format"
].value_counts()

assert format_counts.to_dict() == {
    "Latin": 640,
    "Hangul": 640
}

# 모든 문항에 영문/한글 각각 4회 결과가 있는지 확인
group_counts = results_df.groupby(
    ["question_id", "label_format"]
).size()

assert group_counts.eq(4).all()
assert len(group_counts) == 320

print("Total results:", len(results_df))
print("Unique IDs:", results_df["experiment_id"].nunique())
print("Label counts:")
print(format_counts.to_string())
print("All main experiment validation tests passed.")

In [ ]:
# CSV에서 읽은 True/False 값을 안전하게 변환
results_df["is_valid"] = (
    results_df["is_valid"].astype(str).str.lower() == "true"
)

results_df["is_correct"] = (
    results_df["is_correct"].astype(str).str.lower() == "true"
)

summary = results_df.groupby("label_format").agg(
    total=("is_correct", "size"),
    valid=("is_valid", "sum"),
    correct=("is_correct", "sum"),
    accuracy=("is_correct", "mean")
)

summary["invalid"] = summary["total"] - summary["valid"]

print(summary.to_string())

print("\nSample predictions:")
print(
    results_df[
        [
            "question_id",
            "label_format",
            "rotation",
            "correct_label",
            "raw_response",
            "is_valid",
            "is_correct"
        ]
    ].head(12).to_string(index=False)
)

In [ ]:
import os
import numpy as np
import pandas as pd

RESULT_PATH = "/content/drive/MyDrive/NLP_HW2/results/main_1280_results.csv"

df = pd.read_csv(RESULT_PATH, keep_default_na=False)

# 불리언 자료형 복원
for col in ["is_valid", "is_correct"]:
    df[col] = df[col].astype(str).str.lower().map({
        "true": True,
        "false": False
    })
    assert df[col].notna().all(), f"Invalid values in {col}"

df["predicted_original_id"] = pd.to_numeric(
    df["predicted_original_id"], errors="coerce"
)
df["correct_original_id"] = pd.to_numeric(
    df["correct_original_id"], errors="raise"
)

assert len(df) == 1280
assert df["question_id"].nunique() == 160
assert df["experiment_id"].is_unique

# 정답 판정이 실제 원본 보기 번호와 일치하는지 확인
recomputed = df["predicted_original_id"].eq(
    df["correct_original_id"]
)
assert (recomputed == df["is_correct"]).all()

# 문항별 결과 계산
rows = []

for (qid, fmt), group in df.groupby(
    ["question_id", "label_format"]
):
    assert len(group) == 4
    assert sorted(group["rotation"].tolist()) == [0, 1, 2, 3]

    # 네 번 모두 유효하고 같은 원본 선택지를 고르면 1
    consistent = (
        group["is_valid"].all()
        and group["predicted_original_id"].nunique() == 1
    )

    rows.append({
        "question_id": qid,
        "subject": group["subject"].iloc[0],
        "label_format": fmt,
        "accuracy": group["is_correct"].mean(),
        "consistency": int(consistent),
        "invalid_rate": 1 - group["is_valid"].mean()
    })

item_df = pd.DataFrame(rows)

assert len(item_df) == 320

print("Question-format pairs:", len(item_df))

print("\nMean accuracy and consistency:")
print(
    item_df.groupby("label_format")[
        ["accuracy", "consistency", "invalid_rate"]
    ].mean().to_string()
)

In [ ]:
# 영문과 한글 조건을 문항 ID로 대응시킴
latin = (
    item_df[item_df["label_format"] == "Latin"]
    .set_index("question_id")
    .sort_index()
)

hangul = (
    item_df[item_df["label_format"] == "Hangul"]
    .set_index("question_id")
    .sort_index()
)

assert latin.index.equals(hangul.index)
assert len(latin) == 160

# 한글 - 영문
delta_accuracy = (
    hangul["accuracy"] - latin["accuracy"]
).to_numpy()

delta_consistency = (
    hangul["consistency"] - latin["consistency"]
).to_numpy()

rng = np.random.default_rng(42)

def paired_test(differences, bootstrap_n=2000, signflip_n=10000):
    d = np.asarray(differences, dtype=float)
    n = len(d)

    observed = d.mean()

    # 문항 단위 대응 Bootstrap
    indices = rng.integers(
        0, n, size=(bootstrap_n, n)
    )
    bootstrap_means = d[indices].mean(axis=1)

    ci_low, ci_high = np.quantile(
        bootstrap_means, [0.025, 0.975]
    )

    # 문항 단위 양측 sign-flip randomization test
    signs = rng.choice(
        [-1, 1], size=(signflip_n, n)
    )
    null_means = (signs * d).mean(axis=1)

    p_value = (
        np.sum(np.abs(null_means) >= abs(observed)) + 1
    ) / (signflip_n + 1)

    return {
        "difference": observed,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "p_value": p_value
    }


h1 = paired_test(delta_accuracy)
h2 = paired_test(delta_consistency)

# Holm 보정
def holm_adjust(p_values):
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    adjusted = np.empty(len(p_values))

    running_max = 0.0
    m = len(p_values)

    for rank, idx in enumerate(order):
        value = min(1.0, (m - rank) * p_values[idx])
        running_max = max(running_max, value)
        adjusted[idx] = running_max

    return adjusted


adjusted = holm_adjust([
    h1["p_value"], h2["p_value"]
])

h1["p_holm"] = adjusted[0]
h2["p_holm"] = adjusted[1]

# 결과 표 생성
test_results = pd.DataFrame([
    {"hypothesis": "H1 Accuracy", **h1},
    {"hypothesis": "H2 Consistency", **h2}
])

print(test_results.to_string(index=False))

print("\nInterpretation:")
for _, row in test_results.iterrows():
    outcome = (
        "Significant difference"
        if row["p_holm"] < 0.05
        else "No statistically significant difference"
    )
    print(f"{row['hypothesis']}: {outcome}")

In [ ]:
ANALYSIS_DIR = os.path.join(
    "/content/drive/MyDrive/NLP_HW2", "results"
)

item_df.to_csv(
    os.path.join(ANALYSIS_DIR, "item_level_metrics.csv"),
    index=False,
    encoding="utf-8-sig"
)

test_results.to_csv(
    os.path.join(ANALYSIS_DIR, "hypothesis_tests.csv"),
    index=False,
    encoding="utf-8-sig"
)

print("Analysis results saved.")

## 10. Exploratory analysis and charts

In [ ]:
subject_summary = (
    item_df.groupby(["subject", "label_format"])
    .agg(
        accuracy=("accuracy", "mean"),
        consistency=("consistency", "mean"),
        invalid_rate=("invalid_rate", "mean"),
        questions=("question_id", "nunique")
    )
    .reset_index()
)

for col in ["accuracy", "consistency", "invalid_rate"]:
    subject_summary[col] *= 100

print(subject_summary.to_string(index=False))

subject_summary.to_csv(
    os.path.join(
        ANALYSIS_DIR, "subject_analysis.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
position_summary = (
    df.groupby(["label_format", "correct_position"])
    .agg(
        total=("is_correct", "size"),
        accuracy=("is_correct", "mean")
    )
    .reset_index()
)

position_summary["accuracy"] *= 100

print(position_summary.to_string(index=False))

position_summary.to_csv(
    os.path.join(
        ANALYSIS_DIR, "position_analysis.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
failure_df = df[
    ~df["is_correct"]
].sample(
    n=5,
    random_state=42
)

failure_details = failure_df.merge(
    experiment_df[["experiment_id", "prompt"]],
    on="experiment_id",
    how="left",
    validate="one_to_one"
)

print("Randomly sampled failure IDs:", failure_details["experiment_id"].tolist())

failure_details.to_csv(
    os.path.join(
        ANALYSIS_DIR, "random_failure_examples.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

project = Path("/content/drive/MyDrive/NLP_HW2")
results = project / "results"
figures = project / "figures"

figures.mkdir(exist_ok=True)

# 1. 과목별 정확도
subject_df = pd.read_csv(results / "subject_analysis.csv")

subject_pivot = subject_df.pivot(
    index="subject",
    columns="label_format",
    values="accuracy"
)

ax = subject_pivot[["Latin", "Hangul"]].plot(
    kind="bar",
    figsize=(8, 5)
)

ax.set_ylabel("Accuracy (%)")
ax.set_xlabel("Subject")
ax.set_title("Accuracy by Subject")
plt.xticks(rotation=20)
plt.tight_layout()

plt.savefig(figures / "subject_accuracy.png", dpi=300)
plt.close()

# 2. 정답 위치별 정확도
position_df = pd.read_csv(results / "position_analysis.csv")

position_pivot = position_df.pivot(
    index="correct_position",
    columns="label_format",
    values="accuracy"
)

ax = position_pivot[["Latin", "Hangul"]].plot(
    kind="line",
    marker="o",
    figsize=(8, 5)
)

ax.set_xlabel("Correct Answer Position")
ax.set_ylabel("Accuracy (%)")
ax.set_title("Accuracy by Answer Position")
ax.set_xticks([1, 2, 3, 4])
plt.tight_layout()

plt.savefig(figures / "position_accuracy.png", dpi=300)
plt.close()

# 저장 결과 확인
for file in figures.glob("*.png"):
    print(file.name, file.stat().st_size, "bytes")
print("Figures generated. These plots are exploratory, not causal evidence.")
